### Agenda

- Read multiline nested JSON data
- Inspect nested schema
- Access nested fields directly
- Use `explode` to flatten arrays
- Use `explode_outer` to preserve null arrays
- Add a source column
- Combine DataFrames using `union`
- Apply conditional logic with `when`
- Select clean flattened columns
- Use `pivot` to reshape data
- Review an optional recursive flatten function

### Dataset
- Custom nested JSON file: `coffee_data.json`

### Project Goal
- Practice working with nested JSON data in PySpark
- Flatten arrays and structs into tabular form
- Compare coffee and brewing region data
- Categorize companies
- Summarize results using pivot

In [0]:
from pyspark.sql.functions import explode, explode_outer, col, when, lit, count

In [0]:
file_path = "/Volumes/jenny-demo/my_schema/external_data_source_files/coffee_data.json"

### Read nested JSON file

Use `multiline=True` because the JSON file is formatted across multiple lines.

In [0]:
df = (
    spark.read
    .option("multiline", True)
    .json(file_path)
)

df.printSchema()
display(df)

root
 |-- brewing: struct (nullable = true)
 |    |-- country: struct (nullable = true)
 |    |    |-- company: string (nullable = true)
 |    |    |-- id: long (nullable = true)
 |    |-- region: array (nullable = true)
 |    |    |-- element: struct (containsNull = true)
 |    |    |    |-- id: long (nullable = true)
 |    |    |    |-- name: string (nullable = true)
 |-- coffee: struct (nullable = true)
 |    |-- country: struct (nullable = true)
 |    |    |-- company: string (nullable = true)
 |    |    |-- id: long (nullable = true)
 |    |-- region: array (nullable = true)
 |    |    |-- element: struct (containsNull = true)
 |    |    |    |-- id: long (nullable = true)
 |    |    |    |-- name: string (nullable = true)



brewing,coffee
"List(List(ACME, 2), List(List(1, John Doe), List(3, Jane Smith)))","List(List(ACME, 2), List(List(1, John Doe), List(2, Don Joeh)))"
"List(List(BrewCo, 6), null)","List(List(CoffeeCorp, 5), List(List(4, Anna Lee)))"


### Access a nested field directly

This example pulls:
- `coffee`
- `region`
- first element in the array
- `name`

In [0]:
display(
    df.select(
        col("coffee.region").getItem(0).getField("name").alias("first_coffee_region_name")
    )
)

first_coffee_region_name
John Doe
Anna Lee


### Coffee regions using `explode`

- `explode()` turns each array element into its own row
- null arrays are dropped

In [0]:
coffee_df = df.select(
    explode("coffee.region").alias("region"),
    col("coffee.country.company").alias("company")
)

display(coffee_df)

region,company
"List(1, John Doe)",ACME
"List(2, Don Joeh)",ACME
"List(4, Anna Lee)",CoffeeCorp


### Brewing regions using `explode_outer`

- `explode_outer()` also turns array elements into rows
- unlike `explode()`, it keeps rows where the array is null

In [0]:
brewing_df = df.select(
    explode_outer("brewing.region").alias("region"),
    col("brewing.country.company").alias("company")
)

display(brewing_df)

region,company
"List(1, John Doe)",ACME
"List(3, Jane Smith)",ACME
null,BrewCo


### Add a source column

Track whether each row came from:
- coffee
- brewing

In [0]:
coffee_df = coffee_df.withColumn("source", lit("coffee"))
brewing_df = brewing_df.withColumn("source", lit("brewing"))

display(coffee_df)
display(brewing_df)

region,company,source
"List(1, John Doe)",ACME,coffee
"List(2, Don Joeh)",ACME,coffee
"List(4, Anna Lee)",CoffeeCorp,coffee


region,company,source
"List(1, John Doe)",ACME,brewing
"List(3, Jane Smith)",ACME,brewing
null,BrewCo,brewing


### Combine coffee and brewing data using `union`

Stack both DataFrames together into one result set.

In [0]:
combined_df = coffee_df.union(brewing_df)

display(combined_df)

region,company,source
"List(1, John Doe)",ACME,coffee
"List(2, Don Joeh)",ACME,coffee
"List(4, Anna Lee)",CoffeeCorp,coffee
"List(1, John Doe)",ACME,brewing
"List(3, Jane Smith)",ACME,brewing
null,BrewCo,brewing


### Apply conditional logic using `when`

This is the PySpark equivalent of `CASE WHEN` in SQL.

Rules:
- `ACME` → `Local`
- `BrewCo` → `Partner`
- otherwise → `International`

In [0]:
enriched_df = combined_df.withColumn(
    "company_type",
    when(col("company") == "ACME", lit("Local"))
    .when(col("company") == "BrewCo", lit("Partner"))
    .otherwise(lit("International"))
)

display(enriched_df)

region,company,source,company_type
"List(1, John Doe)",ACME,coffee,Local
"List(2, Don Joeh)",ACME,coffee,Local
"List(4, Anna Lee)",CoffeeCorp,coffee,International
"List(1, John Doe)",ACME,brewing,Local
"List(3, Jane Smith)",ACME,brewing,Local
null,BrewCo,brewing,Partner


### Select clean columns from the exploded struct

The `region` column is still a struct, so now pull out:
- `region.id`
- `region.name`

In [0]:
clean_df = enriched_df.select(
    col("region.id").alias("person_id"),
    col("region.name").alias("person_name"),
    col("company"),
    col("source"),
    col("company_type")
)

display(clean_df)

person_id,person_name,company,source,company_type
1,John Doe,ACME,coffee,Local
2,Don Joeh,ACME,coffee,Local
4,Anna Lee,CoffeeCorp,coffee,International
1,John Doe,ACME,brewing,Local
3,Jane Smith,ACME,brewing,Local
null,null,BrewCo,brewing,Partner


### Pivot to compare employee counts per company

Each company becomes a column.

In [0]:
pivot_df = (
    clean_df.groupBy("person_name")
    .pivot("company")
    .agg(count(lit(1)))
)

display(pivot_df)

person_name,ACME,BrewCo,CoffeeCorp
Anna Lee,null,null,1
John Doe,2,null,null
Don Joeh,1,null,null
Jane Smith,1,null,null
null,null,1,null


### Total people per company

Count how many people are associated with each company.

In [0]:
people_per_company_df = (
    clean_df.groupBy("company")
    .agg(count(lit(1)).alias("total_people"))
)

display(people_per_company_df)

company,total_people
ACME,4
CoffeeCorp,1
BrewCo,1


### Total rows per source

Count how many rows came from:
- coffee
- brewing

In [0]:
people_per_source_df = (
    clean_df.groupBy("source")
    .agg(count(lit(1)).alias("total_people"))
)

display(people_per_source_df)

source,total_people
coffee,3
brewing,3


### Compare `explode` vs `explode_outer`

- `explode` drops null arrays
- `explode_outer` keeps null arrays

In [0]:
print("coffee_df row count:", coffee_df.count())
print("brewing_df row count:", brewing_df.count())

coffee_df row count: 3
brewing_df row count: 3


### Inspect schema fields

This helps show how Spark sees the nested JSON structure.

In [0]:
df.schema.fields

[StructField('brewing', StructType([StructField('country', StructType([StructField('company', StringType(), True), StructField('id', LongType(), True)]), True), StructField('region', ArrayType(StructType([StructField('id', LongType(), True), StructField('name', StringType(), True)]), True), True)]), True),
 StructField('coffee', StructType([StructField('country', StructType([StructField('company', StringType(), True), StructField('id', LongType(), True)]), True), StructField('region', ArrayType(StructType([StructField('id', LongType(), True), StructField('name', StringType(), True)]), True), True)]), True)]

### Optional advanced step: recursive flatten function

This function:
- expands structs into columns
- explodes arrays into rows
- repeats until the DataFrame is flattened

In [0]:
def flatten(df):
    from pyspark.sql.types import StructType, ArrayType
    from pyspark.sql.functions import col, explode_outer

    complex_fields = dict([
        (field.name, field.dataType)
        for field in df.schema.fields
        if isinstance(field.dataType, (ArrayType, StructType))
    ])

    while len(complex_fields) != 0:
        col_name = list(complex_fields.keys())[0]

        if isinstance(complex_fields[col_name], StructType):
            expanded = [
                col(f"{col_name}.{k.name}").alias(f"{col_name}_{k.name}")
                for k in complex_fields[col_name]
            ]
            df = df.select("*", *expanded).drop(col_name)

        elif isinstance(complex_fields[col_name], ArrayType):
            df = df.withColumn(col_name, explode_outer(col_name))

        complex_fields = dict([
            (field.name, field.dataType)
            for field in df.schema.fields
            if isinstance(field.dataType, (ArrayType, StructType))
        ])

    return df

In [0]:
flat_df = flatten(df)

flat_df.printSchema()
display(flat_df)

root
 |-- brewing_country_company: string (nullable = true)
 |-- brewing_country_id: long (nullable = true)
 |-- brewing_region_id: long (nullable = true)
 |-- brewing_region_name: string (nullable = true)
 |-- coffee_country_company: string (nullable = true)
 |-- coffee_country_id: long (nullable = true)
 |-- coffee_region_id: long (nullable = true)
 |-- coffee_region_name: string (nullable = true)



brewing_country_company,brewing_country_id,brewing_region_id,brewing_region_name,coffee_country_company,coffee_country_id,coffee_region_id,coffee_region_name
ACME,2,1,John Doe,ACME,2,1,John Doe
ACME,2,1,John Doe,ACME,2,2,Don Joeh
ACME,2,3,Jane Smith,ACME,2,1,John Doe
ACME,2,3,Jane Smith,ACME,2,2,Don Joeh
BrewCo,6,null,null,CoffeeCorp,5,4,Anna Lee


### Final takeaway

This project practiced:
- reading multiline nested JSON
- inspecting schema
- flattening arrays with `explode`
- preserving null arrays with `explode_outer`
- selecting nested struct fields
- combining DataFrames with `union`
- applying SQL-style conditional logic with `when`
- reshaping results with `pivot`